<a href="https://colab.research.google.com/github/jonaidsharif/Machine-Learning-Projects/blob/main/Sentiment_Analysis_with_BERT_Neural_Network_and_Python.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Install and Import Dependencies

In [2]:
!pip3 install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

Looking in indexes: https://download.pytorch.org/whl/cu118


In [3]:
!pip install transformers requests beautifulsoup4 pandas numpy

In [4]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import requests
from bs4 import BeautifulSoup
import re

# 2. Instantiate Model

In [7]:
tokenizer = AutoTokenizer.from_pretrained('nlptown/bert-base-multilingual-uncased-sentiment')

model = AutoModelForSequenceClassification.from_pretrained('nlptown/bert-base-multilingual-uncased-sentiment')
# https://huggingface.co/nlptown/bert-base-multilingual-uncased-sentiment

# 3. Encode and Calculate Sentiment

In [8]:
tokens = tokenizer.encode('I hated this, absolutely the worst', return_tensors='pt')

In [9]:
result = model(tokens)

In [10]:
result.logits

tensor([[ 4.8750,  1.7880, -0.8356, -3.0027, -2.0727]],
       grad_fn=<AddmmBackward0>)

In [11]:
int(torch.argmax(result.logits))+1

1

In [15]:
import requests
from bs4 import BeautifulSoup
import re

r = requests.get('https://www.yelp.com/biz/mexico-tipico-san-francisco-2?osq=mejico')
soup = BeautifulSoup(r.text, 'html.parser')
#The regex was too restrictive and didn't match any elements
regex = re.compile('comment')
results = soup.find_all('p', {'class':regex})
reviews = [result.text for result in results]

#Check to see if results is empty to avoid the IndexError
if results:
    print(results[0].text)
else:
    print("No matching elements found.")

10/10 Coctel de Camarón. En comparación de los lugares en la cuidad este lugar tiene el coctel con el mejor precio y sabor. In comparison to other places in the city. This spot has the best price and flavor for their shrimp cocktail.


In [17]:
reviews

['10/10 Coctel de Camarón. En comparación de los lugares en la cuidad este lugar tiene el coctel con el mejor precio y sabor. In comparison to other places in the city. This spot has the best price and flavor for their shrimp cocktail.',
 'Pretty good taqueria around the corner from me!* Carnitas burrito - 4.5 stars. Excellent crisp to the carnitas, it comes through even with everything else going on.* Chorizo taco - 4 stars. Again that great crisp! I wish for a little more heartiness to the meat. Each piece is a bit too small.* Fried fish taco - 4 stars. They now have long thin strips that are fried. I thought it was fine, my partner really enjoyed it.* Al Pastor taco - 2.5 stars. Surprisingly boring, meat was rubbery in texture.',
 'I feel like the quality of burritos in SF are declining, but this place did not dissapoint!It gets paccckeedddd... legit waited about 25 minutes in line before ordering and then another 25 for the food.The burritos are big, flavorful, and put together jus

In [20]:
reviews[0]

'10/10 Coctel de Camarón. En comparación de los lugares en la cuidad este lugar tiene el coctel con el mejor precio y sabor. In comparison to other places in the city. This spot has the best price and flavor for their shrimp cocktail.'

# 5. Load Reviews into DataFrame and Score

In [19]:
import numpy as np
import pandas as pd

In [21]:
df = pd.DataFrame(np.array(reviews), columns=['review'])

In [24]:
df['review'].iloc[0] # Changed 'reviews' to 'review'

'10/10 Coctel de Camarón. En comparación de los lugares en la cuidad este lugar tiene el coctel con el mejor precio y sabor. In comparison to other places in the city. This spot has the best price and flavor for their shrimp cocktail.'

In [27]:
def sentiment_score(review):
  tokens = tokenizer.encode(review, return_tensors='pt')
  result = model(tokens) # Fixed the typo 'mdoel' to 'model'
  return int(torch.argmax(result.logits))+1

In [28]:
sentiment_score(df['review'].iloc[1])

4

In [29]:
df['sentiment'] = df['review'].apply(lambda x: sentiment_score(x[:512]))

In [30]:
df

,review,sentiment
0,10/10 Coctel de Camarón. En comparación de los...,5
1,Pretty good taqueria around the corner from me...,4
2,I feel like the quality of burritos in SF are ...,5
3,One of the best taqueria spots in SF! Been goi...,5
4,So first off ! Their food is so good. The gree...,5
5,"The green salsa is some of the best I've had, ...",5
6,"Coming from So Cal (Los Angeles area), but bei...",4
7,"everything is fire. the tacos, huge burritos, ...",5
8,Just recently moved to the area and have been ...,5
9,My orders have lately been wrong. I ask for no...,2


In [31]:
df['review'].iloc[0]

'10/10 Coctel de Camarón. En comparación de los lugares en la cuidad este lugar tiene el coctel con el mejor precio y sabor. In comparison to other places in the city. This spot has the best price and flavor for their shrimp cocktail.'